In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate

In [2]:
# 1. Device and Environment setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model_checkpoint = "distilbert-base-uncased"

In [3]:
# 2. Load Tokenizer and Dataset
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
raw_datasets = load_dataset("stanfordnlp/imdb")

In [4]:
# 3. Parallel Tokenization Process (With proper worker arguments to prevent NameError)
def tokenize_function(examples, tokenizer_object):
    return tokenizer_object(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=128
    )

print("Pre-processing dataset splits...")
tokenized_datasets = raw_datasets.map(
    tokenize_function, 
    batched=True, 
    num_proc=2,
    fn_kwargs={"tokenizer_object": tokenizer}
)

Pre-processing dataset splits...


In [5]:
# 4. Evaluation Metrics Setup
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


In [6]:
# 5. Define Training Configurations
training_args = TrainingArguments(
    output_dir="sentiment_model_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,                       # Reduced to 1 for quick execution verification
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available()            # Dynamic allocation: Only use fp16 if running on a real GPU
)

In [7]:
# 6. Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# 7. Assemble Unified Trainer Engine
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],   # IMDB uses "test" as its evaluation split out-of-the-box
    compute_metrics=compute_metrics,
)

print("\nEngine safely initialized without disconnected variables! Ready to run trainer.train().")


Engine safely initialized without disconnected variables! Ready to run trainer.train().
